# 02 · Silver transformations
Type casting, standardisation, business rules, referential integrity, deduplication and
quarantine. Invalid rows are never corrected: they are written to `quarantine.<entity>` with
`error_reason` and `validation_timestamp`. Valid rows are MERGEd into `silver.<entity>`;
tenants use SCD Type 2.

In [ ]:
%pip install -q pyyaml python-dateutil

In [ ]:
# Setup: make the repo's `src` package importable and select the Databricks environment.
# The notebook runs from <repo>/notebooks inside a Databricks Git folder.
import os, sys
repo_root = os.path.abspath("..")
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)
os.environ["LAKEHOUSE_ENV"] = "databricks"

dbutils.widgets.text("catalog", "workspace")          # Free Edition default; production: real_estate
catalog = dbutils.widgets.get("catalog")
overrides = {"environments": {"databricks": {"catalog": catalog,
                                             "raw_volume_root": f"/Volumes/{catalog}/raw/landing"}}}

from src.pipeline import build_lakehouse
lh, cfg = build_lakehouse(env="databricks", overrides=overrides)
print(f"catalog={cfg.catalog}  as_of={cfg.reporting_as_of}  batches={list(cfg.batches)}")

In [ ]:
from src.pipeline import run_silver
results = run_silver(lh, cfg, "batch_001")     # also runs the Silver data-quality suite
display(spark.createDataFrame([{k: str(v) for k, v in r.items()} for r in results]))

## Why rows were quarantined

In [ ]:
from pyspark.sql import functions as F
frames = []
for t in ["units", "tenants", "leases", "payments", "valuations", "maintenance"]:
    q = spark.table(f"{catalog}.quarantine.{t}")
    frames.append(q.select(F.lit(t).alias("entity"), "batch_id", F.explode(F.split("error_reason", "; ")).alias("reason")))
from functools import reduce
display(reduce(lambda a, b: a.unionByName(b), frames).groupBy("entity", "batch_id", "reason").count().orderBy("entity"))

In [ ]:
display(spark.table(f"{catalog}.ops.dq_results").orderBy(F.desc("checked_at")).limit(40))